# 10 Generate KNN Weighted SOCAT Skill Figure

Purpose: generate the KNN score intermediates used to plot SOCAT skill in the original feature space and DII-weighted feature spaces.

Inputs:
- `../raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr`
- `../raw/socat_mask_feb1982-dec2022.zarr`
- `../intermediates/Finalweights_SOCAT16k_v2_seed13.txt`

Outputs:
- `../intermediates/knn_weighted_scores_seed13_metrics.csv`
- `../intermediates/knn_weighted_scores_seed13_fold_metrics.csv`
- `../intermediates/kNN_weights_models_seed13_metrics.csv`

Method summary:
1. Load SOCAT-masked observations for 2020-2022.
2. Use a 16k sample with random seed 13.
3. Standardize the 13 input features and target `delta_fco2_1D`.
4. Tune kNN once in the original 13-feature space over `n_neighbors = 5, 10, 20` and `weights = uniform, distance`.
5. Reuse the selected kNN settings for every DII-weighted feature count from 4 through 13, evaluated with 5-fold shuffled cross validation.
6. Use row `13 - n_features` of `Finalweights_SOCAT16k_v2_seed13.txt`; zero-weighted features are dropped and retained features are multiplied by their optimized DII weights.

Notes:
- This is a reconstruction notebook derived from the caption and reference image, not an original exported notebook.
- The plotted MAE is in standardized target units, matching the scale of the reference figure.
- The shaded region shows one standard deviation across the 5 cross-validation folds.

Plotting note: `scripts/plot_knn_weighted_figures.py` generates the figure PNGs from these CSV intermediates.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "raw").exists() and (candidate / "intermediates").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root containing raw/ and intermediates/. "
        "Start Jupyter from CO2RepresentCode or one of its subdirectories."
    )

PROJECT_ROOT = find_project_root()
print(f"Using project root: {PROJECT_ROOT}")

RECONSTRUCTION_ZARR = PROJECT_ROOT / "raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr"
SOCAT_MASK_ZARR = PROJECT_ROOT / "raw/socat_mask_feb1982-dec2022.zarr"
WEIGHTS_PATH = PROJECT_ROOT / "intermediates/Finalweights_SOCAT16k_v2_seed13.txt"
METRICS_DIR = PROJECT_ROOT / "intermediates"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TIME_SLICE = slice("2020-01-01", "2022-12-31")
N_SAMPLE = 16000
SAMPLE_SEED = 13
N_SPLITS = 5
CV_RANDOM_STATE = 1
FEATURE_COUNTS = list(range(4, 14))
WEIGHT_TOLERANCE = 1e-12

Using project root: /Users/vivi/Documents/CO2RepresentCode


In [2]:
FEATURES = [
    "sst",
    "sst_anomaly",
    "sss",
    "sss_anomaly",
    "chl_log",
    "chl_log_anomaly",
    "mld_log",
    "xco2_trend",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"

## Load and Scale the SOCAT Sample

This follows the SOCAT DII provenance notebooks: apply the SOCAT mask, define `delta_fco2_1D = fco2 - xco2_trend`, keep 2020-2022, draw the 16k seed-13 sample, then standardize both features and target.

In [3]:
def load_socat_dataframe(reconstruction_zarr, socat_mask_zarr):
    socat_mask = xr.open_dataset(socat_mask_zarr, engine="zarr")
    ds = xr.open_dataset(reconstruction_zarr, engine="zarr")
    ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))

    aligned_mask = socat_mask.reindex(time=ds.time, method="nearest")
    ds = ds.where(aligned_mask.socat_mask == 1)
    ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
    ds = ds[FEATURES + [TARGET]].sel(time=TIME_SLICE)
    return ds.to_dataframe().dropna()

def make_scaled_sample(socat, n_sample=N_SAMPLE, random_state=SAMPLE_SEED):
    sample = socat.sample(n=n_sample, random_state=random_state)
    scaler = StandardScaler()
    scaled = scaler.fit_transform(sample.loc[:, FEATURES + [TARGET]])
    return pd.DataFrame(scaled, columns=FEATURES + [TARGET], index=sample.index)

SOCAT = load_socat_dataframe(RECONSTRUCTION_ZARR, SOCAT_MASK_ZARR)
SCALED = make_scaled_sample(SOCAT)
SCALED.shape

(16000, 14)

## Tune kNN Once in the Original Feature Space

The selected kNN hyperparameters are reused for all weighted feature counts. This follows the figure description and avoids optimizing a separate model at every dimension.

In [4]:
X = SCALED[FEATURES].to_numpy()
y = SCALED[TARGET].to_numpy()

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)

param_grid = {
    "n_neighbors": [5, 10, 20],
    "weights": ["uniform", "distance"],
}

grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=1,
)
grid.fit(X, y)

best_params = grid.best_params_
best_params, grid.best_score_

({'n_neighbors': 5, 'weights': 'distance'}, 0.779996979870889)

In [5]:
def cross_validate_knn(X_features, y_target, params, cv):
    model = KNeighborsRegressor(**params)
    scores = cross_validate(
        model,
        X_features,
        y_target,
        cv=cv,
        scoring=["neg_mean_absolute_error", "r2", "neg_mean_squared_error"],
        return_train_score=True,
        n_jobs=1,
    )
    test_mae = -scores["test_neg_mean_absolute_error"]
    train_mae = -scores["train_neg_mean_absolute_error"]
    test_mse = -scores["test_neg_mean_squared_error"]
    train_mse = -scores["train_neg_mean_squared_error"]
    return {
        "test_mae_folds": test_mae,
        "test_r2_folds": scores["test_r2"],
        "test_mse_folds": test_mse,
        "train_mae_folds": train_mae,
        "train_r2_folds": scores["train_r2"],
        "train_mse_folds": train_mse,
        "mae_test_mean": test_mae.mean(),
        "mae_test_std": test_mae.std(ddof=1),
        "r2_test_mean": scores["test_r2"].mean(),
        "r2_test_std": scores["test_r2"].std(ddof=1),
        "mse_test_mean": test_mse.mean(),
        "mse_test_std": test_mse.std(ddof=1),
        "mae_train_mean": train_mae.mean(),
        "r2_train_mean": scores["train_r2"].mean(),
        "mse_train_mean": train_mse.mean(),
    }

original_scores = cross_validate_knn(X, y, best_params, cv)
{
    "mae_test_mean": original_scores["mae_test_mean"],
    "mae_test_std": original_scores["mae_test_std"],
    "r2_test_mean": original_scores["r2_test_mean"],
    "r2_test_std": original_scores["r2_test_std"],
}

{'mae_test_mean': 0.2696441612844631,
 'mae_test_std': 0.0054817735030204704,
 'r2_test_mean': 0.779996979870889,
 'r2_test_std': 0.013428228279710432}

## Apply DII Weights Across Feature Counts

`Finalweights_SOCAT16k_v2_seed13.txt` has one row per backward-elimination step. Row 0 is the 13-feature metric, row 6 is the 7-feature metric, and in general row `13 - n_features` gives the metric for `n_features`.

In [6]:
dii_weights = np.loadtxt(WEIGHTS_PATH)
assert dii_weights.shape == (len(FEATURES), len(FEATURES)), dii_weights.shape

rows = []
fold_rows = []
for n_features in FEATURE_COUNTS:
    row_index = len(FEATURES) - n_features
    weights = dii_weights[row_index]
    active = np.abs(weights) > WEIGHT_TOLERANCE
    if active.sum() != n_features:
        raise ValueError(
            f"Expected {n_features} active features from row {row_index}, "
            f"found {active.sum()}"
        )

    X_weighted = X[:, active] * weights[active]
    scores = cross_validate_knn(X_weighted, y, best_params, cv)

    rows.append(
        {
            "n_features": n_features,
            "row_index": row_index,
            "active_features": ",".join(np.array(FEATURES)[active]),
            "mae_weighted_mean": scores["mae_test_mean"],
            "mae_weighted_std": scores["mae_test_std"],
            "r2_weighted_mean": scores["r2_test_mean"],
            "r2_weighted_std": scores["r2_test_std"],
            "mse_weighted_mean": scores["mse_test_mean"],
            "mse_weighted_std": scores["mse_test_std"],
            "mae_original_13d_mean": original_scores["mae_test_mean"],
            "mae_original_13d_std": original_scores["mae_test_std"],
            "r2_original_13d_mean": original_scores["r2_test_mean"],
            "r2_original_13d_std": original_scores["r2_test_std"],
            **{f"knn_{key}": value for key, value in best_params.items()},
        }
    )

    for fold, (mae, r2, mse) in enumerate(
        zip(scores["test_mae_folds"], scores["test_r2_folds"], scores["test_mse_folds"]),
        start=1,
    ):
        fold_rows.append(
            {
                "n_features": n_features,
                "fold": fold,
                "mae_weighted": mae,
                "r2_weighted": r2,
                "mse_weighted": mse,
            }
        )

metrics = pd.DataFrame(rows)
fold_metrics = pd.DataFrame(fold_rows)
metrics.to_csv(METRICS_DIR / "knn_weighted_scores_seed13_metrics.csv", index=False)
fold_metrics.to_csv(METRICS_DIR / "knn_weighted_scores_seed13_fold_metrics.csv", index=False)
metrics

,n_features,row_index,active_features,mae_weighted_mean,mae_weighted_std,r2_weighted_mean,r2_weighted_std,mse_weighted_mean,mse_weighted_std,mae_original_13d_mean,mae_original_13d_std,r2_original_13d_mean,r2_original_13d_std,knn_n_neighbors,knn_weights
0,4,9,"sst,A,B,C",0.316379,0.007846,0.698490,0.018441,0.301441,0.021565,0.269644,0.005482,0.779997,0.013428,5,distance
1,5,8,"sst,sss,A,B,C",0.299464,0.009368,0.718404,0.019840,0.281636,0.023781,0.269644,0.005482,0.779997,0.013428,5,distance
2,6,7,"sst,sss,A,B,C,T1",0.257464,0.005864,0.799668,0.008127,0.200330,0.012220,0.269644,0.005482,0.779997,0.013428,5,distance
3,7,6,"sst,sss,A,B,C,T0,T1",0.236012,0.005430,0.826950,0.007277,0.173117,0.012211,0.269644,0.005482,0.779997,0.013428,5,distance
4,8,5,"sst,sss,mld_log,A,B,C,T0,T1",0.235489,0.005913,0.828707,0.007651,0.171316,0.011479,0.269644,0.005482,0.779997,0.013428,5,distance
5,9,4,"sst,sss,sss_anomaly,mld_log,A,B,C,T0,T1",0.234437,0.006000,0.829095,0.004957,0.171002,0.011367,0.269644,0.005482,0.779997,0.013428,5,distance
6,10,3,"sst,sss,sss_anomaly,chl_log,mld_log,A,B,C,T0,T1",0.228371,0.006363,0.840289,0.006764,0.159827,0.012117,0.269644,0.005482,0.779997,0.013428,5,distance
7,11,2,"sst,sss,sss_anomaly,chl_log,mld_log,xco2_trend...",0.221694,0.005374,0.844122,0.005094,0.156021,0.011609,0.269644,0.005482,0.779997,0.013428,5,distance
8,12,1,"sst,sst_anomaly,sss,sss_anomaly,chl_log,mld_lo...",0.222001,0.005345,0.843922,0.004996,0.156217,0.011510,0.269644,0.005482,0.779997,0.013428,5,distance
9,13,0,"sst,sst_anomaly,sss,sss_anomaly,chl_log,chl_lo...",0.222452,0.005376,0.843261,0.004912,0.156889,0.011675,0.269644,0.005482,0.779997,0.013428,5,distance


## Plot the Figure

## KNN Skill Using SOCAT and GOBM-Derived Weights

This section reconstructs the companion panel `kNN_weights_models_seed13.png`.
It evaluates the same SOCAT 16k seed-13 sample and the same kNN hyperparameters selected above, but swaps the DII metric source:

- SOCAT weights: `Finalweights_SOCAT16k_v2_seed13.txt`
- GOBM weights in the SOCAT mask: `FromRomy/Finalweights_sampled1_seed13_<model>.txt`

For each weight source and each feature count from 4 through 13, row `13 - n_features` of the corresponding weight matrix is applied to the SOCAT features, and the mean test MAE is estimated with the same 5-fold shuffled cross validation.

In [8]:
GOBM_MODELS = ["ETHZ", "FESOM", "NorESM", "MRI", "IPSL"]
GOBM_COLORS = {
    "SOCAT": "#35cbe8",
    "ETHZ": "blue",
    "FESOM": "red",
    "NorESM": "green",
    "MRI": "black",
    "IPSL": "magenta",
}

WEIGHT_SOURCES = {
    "SOCAT": WEIGHTS_PATH,
    **{
        model: PROJECT_ROOT / f"intermediates/FromRomy/Finalweights_sampled1_seed13_{model}.txt"
        for model in GOBM_MODELS
    },
}

def evaluate_weight_source(weight_source, weights_path):
    weights_matrix = np.loadtxt(weights_path)
    if weights_matrix.shape != (len(FEATURES), len(FEATURES)):
        raise ValueError(f"Unexpected shape for {weights_path}: {weights_matrix.shape}")

    rows = []
    for n_features in FEATURE_COUNTS:
        row_index = len(FEATURES) - n_features
        weights = weights_matrix[row_index]
        active = np.abs(weights) > WEIGHT_TOLERANCE
        if active.sum() != n_features:
            raise ValueError(
                f"{weight_source}: expected {n_features} active features from row "
                f"{row_index}, found {active.sum()}"
            )

        X_weighted = X[:, active] * weights[active]
        scores = cross_validate_knn(X_weighted, y, best_params, cv)
        rows.append(
            {
                "weight_source": weight_source,
                "n_features": n_features,
                "row_index": row_index,
                "active_features": ",".join(np.array(FEATURES)[active]),
                "mae_mean": scores["mae_test_mean"],
                "mae_std": scores["mae_test_std"],
                "r2_mean": scores["r2_test_mean"],
                "r2_std": scores["r2_test_std"],
                "weights_path": str(weights_path),
                **{f"knn_{key}": value for key, value in best_params.items()},
            }
        )
    return rows

model_weight_rows = []
for weight_source, weights_path in WEIGHT_SOURCES.items():
    if not weights_path.exists():
        raise FileNotFoundError(weights_path)
    model_weight_rows.extend(evaluate_weight_source(weight_source, weights_path))

model_weight_metrics = pd.DataFrame(model_weight_rows)
model_weight_metrics.to_csv(METRICS_DIR / "kNN_weights_models_seed13_metrics.csv", index=False)
model_weight_metrics.head()

,weight_source,n_features,row_index,active_features,mae_mean,mae_std,r2_mean,r2_std,weights_path,knn_n_neighbors,knn_weights
0,SOCAT,4,9,"sst,A,B,C",0.316379,0.007846,0.698490,0.018441,/Users/vivi/Documents/CO2RepresentCode/interme...,5,distance
1,SOCAT,5,8,"sst,sss,A,B,C",0.299464,0.009368,0.718404,0.019840,/Users/vivi/Documents/CO2RepresentCode/interme...,5,distance
2,SOCAT,6,7,"sst,sss,A,B,C,T1",0.257464,0.005864,0.799668,0.008127,/Users/vivi/Documents/CO2RepresentCode/interme...,5,distance
3,SOCAT,7,6,"sst,sss,A,B,C,T0,T1",0.236012,0.005430,0.826950,0.007277,/Users/vivi/Documents/CO2RepresentCode/interme...,5,distance
4,SOCAT,8,5,"sst,sss,mld_log,A,B,C,T0,T1",0.235489,0.005913,0.828707,0.007651,/Users/vivi/Documents/CO2RepresentCode/interme...,5,distance
